<a href="https://colab.research.google.com/github/Steve-Falkovsky/Hypencoder-Entity-Linking/blob/main/notebooks/Hypencoder_Vs_fine_tuned_Hypencoder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- Environment lock (run once per fresh runtime) ---
# because libraries like transformers can change and make your code non-runnable :)
# Torch updates also broke the code but installing it in colab to an older version
# takes 2 minutes every time so I just solved the one bug

!wget -q https://raw.githubusercontent.com/Steve-Falkovsky/Hypencoder-Entity-Linking/main/colab_utils/env_setup.py
import env_setup
env_setup.install_env()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.0/86.0 kB 2.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 33.3 MB/s eta 0:00:00
Installed. Restarting runtime is required.


In [ ]:
import os

REPO_NAME = "Hypencoder-Entity-Linking"
GIT_URL = f"https://github.com/Steve-Falkovsky/{REPO_NAME}.git"
BRANCH_NAME = "main"

!git clone -b {BRANCH_NAME} --single-branch {GIT_URL}

# Move into the downloaded repo (The Root)
os.chdir(REPO_NAME)

%pip install -q -e "./hypencoder-paper"

os.chdir("hypencoder-paper")

print(f"📍 Working Directory is now: {os.getcwd()}")
print("✅ Environment Ready!")

Cloning into 'Hypencoder-Entity-Linking'...
remote: Enumerating objects: 455, done.
remote: Counting objects: 100% (88/88), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 455 (delta 50), reused 39 (delta 23), pack-reused 367 (from 1)
Receiving objects: 100% (455/455), 966.20 KiB | 5.08 MiB/s, done.
Resolving deltas: 100% (241/241), done.
  Preparing metadata (setup.py) ... done
📍 Working Directory is now: /content/Hypencoder-Entity-Linking/hypencoder-paper
✅ Environment Ready!


### Load the model

In [ ]:
# Core Hypencoder model for outputing dense vector representations
from hypencoder_cb.modeling.hypencoder import Hypencoder, HypencoderDualEncoder, TextEncoder
from transformers import AutoTokenizer

model_name = "Stevenf232/SapBERT_freeze_hypencoder_context"
# model_name = "Stevenf232/SapBERT_freeze_hypencoder_hardneg"

dual_encoder = HypencoderDualEncoder.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)


query_encoder: Hypencoder = dual_encoder.query_encoder
passage_encoder: TextEncoder = dual_encoder.passage_encoder

config.json:   0%|          | 0.00/925 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/483M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/462 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

### Move the model to the GPU

In [ ]:
import torch

# Setup the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")  # This should say 'cuda'

# Move the model to the GPU
passage_encoder.to(device)
query_encoder.to(device)

Using device: cuda


Hypencoder(
  (transformer): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, element

### Load datasets and tokenise

In [ ]:
from datasets import load_dataset

# there are all "positive" pairs
dataset = load_dataset("Stevenf232/BC5CDR_MeSH2015_complete")

train.jsonl: 0.00B [00:00, ?B/s]

validation.jsonl: 0.00B [00:00, ?B/s]

test.jsonl: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/5424 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5445 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5516 [00:00<?, ? examples/s]

In [ ]:
test_data = dataset['test']
print(test_data)

mention_names = test_data['mention']
mention_texts = test_data['mention_text']
entity_names = test_data['entity']
definitions = test_data['definition']

print(mention_names[:3])
print(entity_names[:3])

Dataset({
    features: ['mention', 'mention_text', 'entity', 'aliases', 'definition', 'id'],
    num_rows: 5516
})
['Famotidine', 'delirium', 'Famotidine']
['Famotidine', 'Delirium', 'Famotidine']


In [ ]:
test_data[0]

{'mention': 'Famotidine',
 'mention_text': 'Famotidine-associated delirium. A series of six cases.',
 'entity': 'Famotidine',
 'aliases': 'Famotidine Hydrochloride MK 208 MK-208 MK208 Pepcid YM 11170 YM-11170 YM11170',
 'definition': 'A competitive histamine H2-receptor antagonist. Its main pharmacodynamic effect is the inhibition of gastric secretion.\n    ',
 'id': 'MESH:D015738'}

In [ ]:
# util function to get context window around the mention in the text
!wget -q https://raw.githubusercontent.com/Steve-Falkovsky/Hypencoder-Entity-Linking/main/colab_utils/text_preprocessing.py
from text_preprocessing import get_mt_window

# reduce the mention text and definition to the first 128 chars
window = [get_mt_window(mention,text) for mention, text in zip(mention_names, mention_texts)]
short_def = [text[:128] for text in definitions]

In [ ]:
# convert from type "datasets" to python list
queries = list(zip(mention_names, window))

passages = list(zip(entity_names, short_def))

# the output of the tokenizer contains 3 fields:
# input_ids, token_type_ids, and attention_mask
# all contain a tensor in the shape (number of queries, max number of tokens)

query_inputs = tokenizer(queries, return_tensors="pt", padding=True, truncation=True, max_length = 128)
passage_inputs = tokenizer(passages, return_tensors="pt", padding=True, truncation=True, max_length = 128)


In [ ]:
print(f"query_inputs:\n{query_inputs}")
print("\n\n\n")
print(f"passage_inputs:\n{passage_inputs}")

query_inputs:
{'input_ids': tensor([[    2,  3011,  4090,  ...,     0,     0,     0],
        [    2, 18166,     3,  ...,     0,     0,     0],
        [    2,  3011,  4090,  ...,     0,     0,     0],
        ...,
        [    2, 17632,     3,  ...,     0,     0,     0],
        [    2,  3817, 22750,  ...,     0,     0,     0],
        [    2,  2041,     3,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}




passage_inputs:
{'input_ids': tensor([[    2,  3011,  4090,  ...,     0,     0,     0],
        [    2, 18166,     3,  ...,     0, 

# Passage Encodings


In [ ]:
from tqdm import tqdm
import torch
from torch.amp import autocast

def batch_encode_passages(encoder ,passages):
  batch_size=256
  entity_name_features = []

  num_passages = passages["input_ids"].shape[0]

  with torch.no_grad(): # Disable gradient calculation (saves tons of memory)
    for i in tqdm(range(0, num_passages, batch_size), desc="Extracting features"):

        # extract entity features
        # Autocast does the math in fp16 where possible (default is fp32)
        # this will save memory and increase speed. The loss in precision shouldn't matter much (can check on a small sample if we want)
        with autocast("cuda"):
          features = encoder(
              input_ids=passages["input_ids"][i:i + batch_size].to(device),
              attention_mask=passages["attention_mask"][i:i + batch_size].to(device)
            ).representation

          entity_name_features.append(features.detach().cpu()) # Detach and move to CPU to save VRAM/RAM


  features_tensor = torch.cat(entity_name_features, dim=0)

  return features_tensor

In [ ]:
passage_embeddings = batch_encode_passages(passage_encoder, passage_inputs)

Extracting features: 100%|██████████| 22/22 [00:04<00:00,  5.26it/s]


##Now, we create the q-nets.

For each q-net, we feed through it all the passages the calculate the similarity.

But the q-nets are created in batches, and every batch is represented as a single object `NoTorchSequential`. (Check out the `RepeatedDenseBlockConverter` class in q_net.py for more info)

This object expects an input in the shape (N, M, H):

* N = number of queries (mentions)

* M = number of passages (entities)

* H = Hidden dimension (e.g., 768 for BERT)



---



The passage embeddings have the shape (M, H) so we must create an additional dimension of size N.

This will be done like so:
`passages_batch = passages.unsqueeze(0).expand(num_queries, -1, -1)`

* `.unsqueeze()` adds a new dimension (in our case at location 0)

* `.expand()` "expands" that new dimension to be size "num_queries"

* `.expand()` creates a view, so it costs almost 0 memory! (compared to .repeat() which changes the tensor)

# Q-nets take **a lot** of memory.

Instead of creating all of them and then doing the similarity calculation, we will create batches and calculate similarities for just those q-nets, then discard those q-nets and move on to the next batch.

In [ ]:
def batch_encode_queries(encoder, queries, passage_embeddings):
  batch_size = 8
  similarity_scores = []

  num_queries = queries["input_ids"].shape[0]

  with torch.no_grad():
    for i in tqdm(range(0, num_queries, batch_size), desc="Creating q-nets and calculating similarity scores"):

        # create q-nets
        with autocast("cuda"):
          q_nets = encoder(
              input_ids=queries["input_ids"][i:i + batch_size].to(device),
              attention_mask=queries["attention_mask"][i:i + batch_size].to(device)
            ).representation


        passages_gpu = passage_embeddings.to(device)

        # Note: we use q_nets.num_queries (our repo's noTorch equivalent of q_nets.shape[0]) instead of batch_size
        # because the total number might not be divisible by batch_size so the last batch might be smaller than the actual batch size
        passages_batch = passages_gpu.unsqueeze(0).expand(q_nets.num_queries, -1, -1)

        # calculate similarity
        batch_scores = q_nets(passages_batch)
        similarity_scores.append(batch_scores.detach().cpu())


  scores_tensor = torch.cat(similarity_scores, dim=0)
  return scores_tensor


In [ ]:
similarity_scores = batch_encode_queries(query_encoder, query_inputs, passage_embeddings)

Creating q-nets and calculating similarity scores: 100%|██████████| 690/690 [00:48<00:00, 14.33it/s]


In [ ]:
# Case 1 - comparing a query to its respective passage

# In the simple case where each q_net only takes one passage, we can just
# reshape the passage_embeddings to (N, 1, H).
# passage_embeddings_single = passage_embeddings.unsqueeze(1)
# print(f"passage_embeddings shape: {passage_embeddings_single.shape}")
# giving the nueral network the input of passage_embeddings
# the output provides the relevance score of query 1 against passage 1, query 2 against passage 2, etc...
# scores = q_nets(passage_embeddings_single)
# print(f"scores: {scores}")

In [ ]:
# Case 2 - comparing a query to all passages

# The case where each q_net takes multiple passages
# meaning multiple passages are now associated with each of the queries

# this operation creates a 3D tensor which takes too much memory
# passage_embeddings_multi = passage_embeddings.repeat(N, 1).reshape(N, M, H)
# print(f"passage_embeddings shape: {passage_embeddings_multi.shape}")


# unbatched similarity scores for q-nets
# similarity_scores = q_nets(passage_embeddings_multi)
# print(f"similarity_scores shape: {similarity_scores.shape}")
#print(f"similarity_scores: {similarity_scores}")


In [ ]:
similarity_scores

tensor([[[18.6268],
         [ 6.2527],
         [18.6268],
         ...,
         [-0.7267],
         [ 1.7306],
         [ 1.9360]],

        [[14.0873],
         [16.5782],
         [14.0873],
         ...,
         [-2.3481],
         [-0.4564],
         [ 3.3655]],

        [[17.7600],
         [-3.4173],
         [17.7600],
         ...,
         [-0.1001],
         [ 1.7571],
         [-1.9261]],

        ...,

        [[-1.0074],
         [ 6.8022],
         [-1.0074],
         ...,
         [13.5962],
         [ 7.6960],
         [13.2858]],

        [[-0.0375],
         [ 0.9741],
         [-0.0375],
         ...,
         [ 0.7041],
         [11.5765],
         [13.1784]],

        [[ 1.4008],
         [ 5.6855],
         [ 1.4008],
         ...,
         [ 1.7419],
         [11.3533],
         [17.3875]]])

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def evaluate(test_data):
  correct_count = 0
  top_idxs = torch.argmax(similarity_scores,dim=1).flatten()

  for i in range(len(queries)):
      top_idx = top_idxs[i]
      # the conversion to int from here on out is because the original idx is of type numpy.int64
      top_match_id = test_data["id"][int(top_idx)]
      correct_id = test_data["id"][int(i)]

      if top_match_id == correct_id:
          correct_count += 1

      mention_name = test_data["mention"][int(i)]
      top_match = test_data["entity"][int(top_idx)]
      correct_name = test_data["entity"][int(i)]
      seen_mention_text = window[int(i)]
      seen_definition = short_def[int(i)]
      mention_text = test_data["mention_text"][int(i)]
      definition = test_data["definition"][int(i)]
      print(f"""
            mention_name: {mention_name}
            correct entity name: {correct_name}
            top_match: {top_match}
            seen_mention_text: {seen_mention_text}
            seen_definition: {seen_definition}
            mention_text: {mention_text}
            definition: {definition}
            """)


  print(f"total comparisons: {len(queries)}")
  print(f"correct comparisons: {correct_count}")
  print(f"accuracy: {correct_count / len(queries)}")

In [ ]:
evaluate(test_data)

Streaming output truncated to the last 5000 lines.

            mention_name: isoprenaline
            correct entity name: Isoproterenol
            top_match: Isoproterenol
            seen_mention_text: n blocking the positive inotropic and chronotropic responses to isoprenaline, (+)-propranolol had less than one hundredth the potency of (-)
            seen_definition: Isopropyl analog of EPINEPHRINE; beta-sympathomimetic that acts on the heart, bronchi, skeletal muscle, alimentary tract, etc. I
            mention_text: 1. The optical isomers of propranolol have been compared for their beta-blocking and antiarrhythmic activities.2. In blocking the positive inotropic and chronotropic responses to isoprenaline, (+)-propranolol had less than one hundredth the potency of (-)-propranolol. At dose levels of (+)-propranolol which attenuated the responses to isoprenaline, there was a significant prolongation of the PR interval of the electrocardiogram.3. The metabolic responses to isopren

In [ ]:
# more evaluation methods
#from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score
# print(f"{accuracy_score(train_labels, predicted_labels)=:.3f}")
# print(f"{recall_score(train_labels, predicted_labels)=:.3f}")
# print(f"{precision_score(train_labels, predicted_labels)=:.3f}")
# print(f"{f1_score(train_labels, predicted_labels)=:.3f}")